# LangChain Hands-on 15 — Callbacks, Caching, Rate Limits & LangSmith
**Trainer:** Ajeetkumar

**What we will practise**
- Callbacks: see what happens inside (tokens, time)
- LLM cache: same question → no new API call
- Rate limiter: stay inside free-tier limits
- Async & concurrency
- LangSmith tracing (debug & monitor)

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-groq

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

## 1. Callbacks — hooks that run on LLM events

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
import time

In [ ]:
class MyLogger(BaseCallbackHandler):
    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.start = time.time()
        print("🚀 LLM started")
    def on_llm_end(self, response, **kwargs):
        usage = response.generations[0][0].message.usage_metadata
        print(f"✅ LLM finished in {time.time() - self.start:.2f}s | tokens: {usage}")
    def on_llm_error(self, error, **kwargs):
        print("❌ error:", error)

In [ ]:
llm.invoke("Tell me a fun fact about India", config={"callbacks": [MyLogger()]}).content

### Streaming tokens with a callback

In [ ]:
class PrintTokens(BaseCallbackHandler):
    def on_llm_new_token(self, token, **kwargs):
        print(token, end="", flush=True)

streaming_llm = ChatGroq(model="openai/gpt-oss-20b", streaming=True, callbacks=[PrintTokens()])
_ = streaming_llm.invoke("Write 3 lines about the Himalayas")

## 2. Cache — save money on repeated questions

In [ ]:
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

In [ ]:
set_llm_cache(InMemoryCache())

In [ ]:
t = time.time(); llm.invoke("What is the capital of Japan?"); print("1st call:", round(time.time() - t, 2), "s")
t = time.time(); llm.invoke("What is the capital of Japan?"); print("2nd call:", round(time.time() - t, 4), "s  (from cache)")

In [ ]:
set_llm_cache(None)     # switch off

> For a cache that survives restarts: `from langchain_community.cache import SQLiteCache` → `set_llm_cache(SQLiteCache("cache.db"))`

## 3. Rate limiter — don't hit the provider's RPM limit

In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter

In [ ]:
rate_limiter = InMemoryRateLimiter(requests_per_second=0.5, check_every_n_seconds=0.1, max_bucket_size=1)   # 1 call every 2 s
limited_llm = ChatGroq(model="openai/gpt-oss-20b", rate_limiter=rate_limiter)

In [ ]:
for i in range(3):
    t = time.time()
    limited_llm.invoke("hi")
    print(f"call {i + 1} took {time.time() - t:.2f}s")

## 4. Async + batch concurrency

In [ ]:
import asyncio
questions = [f"Give one fact about number {i}" for i in range(1, 6)]

In [ ]:
t = time.time()
answers = await asyncio.gather(*[llm.ainvoke(q) for q in questions])
print("async total:", round(time.time() - t, 2), "s")

In [ ]:
answers = llm.batch(questions, config={"max_concurrency": 2})      # max 2 at a time
len(answers)

## 5. LangSmith — trace & debug every call
1. Create a free account at **smith.langchain.com** → Settings → API Keys
2. Save it in Colab Secrets as `langsmith_key`
3. Run the cell below — every chain / agent run appears in the LangSmith dashboard (inputs, outputs, tokens, latency, errors).

In [ ]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("langsmith_key")
os.environ["LANGSMITH_PROJECT"] = "langchain-handson"

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = ChatPromptTemplate.from_template("Explain {topic} in 2 lines") | llm | StrOutputParser()
chain.invoke({"topic": "vector database"}, config={"run_name": "explain-topic", "tags": ["demo"], "metadata": {"user": "ajeet"}})

# Production checklist
- ✅ API keys in secrets, never in code
- ✅ timeout, max_retries, fallbacks
- ✅ cache repeated calls, rate-limit on free tiers
- ✅ trace with LangSmith, log tokens & cost
- ✅ test prompts on a fixed set of questions before changing them